# 시간 가중 벡터 검색(Time-weighted retrieval)

의미 유사도와 최근 접근 시점을 결합해 검색합니다. classic의
`TimeWeightedVectorStoreRetriever` 및 전역 시간을 바꾸는 `mock_now` 대신,
현재 시각을 함수 인자로 주입하는 작은 인덱스를 만들어 계산과 테스트를 명확히 합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-core==1.6.3" "langchain-openai==1.6.2" \
#   "langchain-chroma==1.1.0" python-dotenv


In [ ]:
import getpass
import os
from datetime import datetime, timedelta, timezone
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


def bounded_cosine_relevance(distance: float) -> float:
    return max(0.0, min(1.0, 1.0 - distance / 2.0))


## 점수 정의

이 예제의 결합 점수는 다음과 같습니다.

$$score = semantic\_relevance + (1-decay\_rate)^{hours\_since\_access}$$

`decay_rate`가 0에 가까우면 오래된 정보도 최근성 점수를 오래 유지하고,
1에 가까우면 최근성 점수가 빠르게 0으로 감소합니다. 의미 점수와 최근성 점수의
범위가 다를 수 있으므로 운영에서는 평가 데이터로 가중치와 보정을 추가하세요.


In [ ]:
class TimeWeightedIndex:
    def __init__(self, *, decay_rate: float, k: int = 1, fetch_k: int = 8):
        if not 0 <= decay_rate <= 1:
            raise ValueError("decay_rate는 0과 1 사이여야 합니다.")
        self.decay_rate = decay_rate
        self.k = k
        self.fetch_k = fetch_k
        self.documents: dict[str, Document] = {}
        self.last_accessed: dict[str, datetime] = {}
        self.vectorstore = Chroma(
            collection_name=f"time-weighted-{uuid4().hex}",
            embedding_function=embeddings,
            collection_configuration=CHROMA_CONFIGURATION,
            relevance_score_fn=bounded_cosine_relevance,
        )

    def add_documents(
        self,
        documents: list[Document],
        *,
        now: datetime | None = None,
    ) -> None:
        current = now or datetime.now(timezone.utc)
        prepared: list[Document] = []
        ids: list[str] = []
        for doc in documents:
            memory_id = str(doc.metadata.get("memory_id", uuid4().hex))
            accessed = doc.metadata.get("last_accessed_at", current)
            if isinstance(accessed, str):
                accessed = datetime.fromisoformat(accessed)
            if accessed.tzinfo is None:
                accessed = accessed.replace(tzinfo=timezone.utc)

            stored = Document(
                page_content=doc.page_content,
                metadata={
                    **doc.metadata,
                    "memory_id": memory_id,
                    "last_accessed_at": accessed.isoformat(),
                },
            )
            self.documents[memory_id] = stored
            self.last_accessed[memory_id] = accessed
            prepared.append(stored)
            ids.append(memory_id)
        self.vectorstore.add_documents(prepared, ids=ids)

    def search(
        self,
        query: str,
        *,
        now: datetime | None = None,
    ) -> list[Document]:
        current = now or datetime.now(timezone.utc)
        candidate_count = min(self.fetch_k, len(self.documents))
        if candidate_count == 0:
            return []
        candidates = self.vectorstore.similarity_search_with_relevance_scores(
            query,
            k=candidate_count,
        )

        scored: list[tuple[float, str, float, float]] = []
        for doc, semantic_score in candidates:
            memory_id = doc.metadata["memory_id"]
            elapsed = current - self.last_accessed[memory_id]
            hours = max(0.0, elapsed.total_seconds() / 3_600)
            recency_score = (1.0 - self.decay_rate) ** hours
            combined_score = float(semantic_score) + recency_score
            scored.append(
                (combined_score, memory_id, float(semantic_score), recency_score)
            )

        scored.sort(key=lambda item: item[0], reverse=True)
        results: list[Document] = []
        for combined, memory_id, semantic, recency in scored[: self.k]:
            original = self.documents[memory_id]
            previous_access = self.last_accessed[memory_id]
            results.append(
                Document(
                    page_content=original.page_content,
                    metadata={
                        **original.metadata,
                        "last_accessed_at_before": previous_access.isoformat(),
                        "semantic_score": semantic,
                        "recency_score": recency,
                        "combined_score": combined,
                    },
                )
            )
            self.last_accessed[memory_id] = current
        return results


## 낮은 감쇠율과 높은 감쇠율 비교


In [ ]:
reference_time = datetime.now(timezone.utc)
memories = [
    Document(
        page_content="테디노트 구독해 주세요.",
        metadata={"last_accessed_at": reference_time - timedelta(days=1)},
    ),
    Document(
        page_content="테디노트 구독 해주실꺼죠? Please!",
        metadata={"last_accessed_at": reference_time},
    ),
]


def make_demo(decay_rate: float) -> tuple[TimeWeightedIndex, RunnableLambda]:
    index = TimeWeightedIndex(decay_rate=decay_rate, k=1, fetch_k=2)
    index.add_documents(memories, now=reference_time)
    runnable = RunnableLambda(
        lambda request: index.search(request["query"], now=request.get("now"))
    ).with_config({"run_name": f"time_weighted_{decay_rate}"})
    return index, runnable


low_index, low_decay_retriever = make_demo(0.000001)
high_index, high_decay_retriever = make_demo(0.999)

low_result = low_decay_retriever.invoke(
    {"query": "테디노트", "now": reference_time}
)
high_result = high_decay_retriever.invoke(
    {"query": "테디노트", "now": reference_time}
)

print("낮은 감쇠율:", low_result[0].page_content, low_result[0].metadata)
print("높은 감쇠율:", high_result[0].page_content, high_result[0].metadata)


## 가상 시간 테스트

전역 `datetime.now()`를 패치하지 않고 `now`를 주입하므로 테스트가 다른 셀이나
라이브러리에 영향을 주지 않습니다.


In [ ]:
future_time = reference_time + timedelta(days=7)
future_result = high_decay_retriever.invoke(
    {"query": "테디노트", "now": future_time}
)
print(future_result[0].page_content)
print(future_result[0].metadata)
